# vbOCP - Test 2 (delfino): pipeline completa

Stessa pipeline di `run_pipeline.ipynb` (Test 1) applicata al dominio con buco a forma di delfino (Strazzullo & Vicini 2023, Fig. 9). Notebook orchestratore bash-style: nessuna logica del modello qui dentro, solo chiamate agli script di `src/`. Tutto cio' che cambia rispetto al Test 1 sta in `configs/delfino.yaml` (mesh, marker di bordo, bordo di controllo `x_ctrl = mu_u`, range di `mu_u`).

Dati e output in `data/delfino/` e modelli con prefisso `delfino_`, cosi' non si sovrascrive nulla del Test 1.

**Contenuto:**
1. Setup
2. FOM
3. ROM (POD, PODNN)
4. GNN (gca-rom)
5. Controllo (POD)

## 0. Setup Colab (solo su Colab, in locale viene saltato)

Installa le dipendenze e clona/aggiorna la repo da GitHub: serve aver fatto `git push` di mesh, config e codice del delfino.

In [ ]:
import sys

if 'google.colab' in sys.modules:
    GITHUB_URL = 'https://github.com/roccnroll/vbOCP.git'
    REPO_DIR = 'vbOCP'

    !pip install -q pypolydim gmsh
    import os
    if not os.path.isdir(REPO_DIR):
        !git clone {GITHUB_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull
    os.chdir(REPO_DIR)
    print('cwd:', os.getcwd())
else:
    print('Non su Colab: setup saltato')

## 1. Setup

In [ ]:
import os

# si sposta nella root della repo (dove c'e' src/), indipendentemente da dove parte il kernel
while not os.path.isdir('src') and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir('..')
assert os.path.isdir('src'), "src/ non trovata: verifica dove e' montata la repo vbOCP"

TEST           = 'delfino'
REPO_ROOT      = os.getcwd()
CONFIG_PATH    = os.path.join('configs', f'{TEST}.yaml')
DATA_DIR       = os.path.join('data', TEST)
SNAPSHOTS_DIR  = os.path.join(DATA_DIR, 'snapshots')
OUTPUT_DIR     = os.path.join('notebooks', 'output', TEST)

assert os.path.isdir('data/meshes/delfino_csv'), "mesh mancante: python src/full_order/convert_dolphin_mesh.py"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SNAPSHOTS_DIR, exist_ok=True)

print('REPO_ROOT    :', REPO_ROOT)
print('CONFIG_PATH  :', CONFIG_PATH)
print('SNAPSHOTS_DIR:', SNAPSHOTS_DIR)

## 2. FOM

### 2.1 Solve singolo

Confronta con la Fig. 11 del paper (`mu1=12, mu2=2.5`, `mu_u` in {0.3, 0.5, 0.7}).

In [ ]:
!python -m src.full_order.run_single_solve \
    --config {CONFIG_PATH} \
    --mu1 12 \
    --mu2 2.5 \
    --mu_u 0.5 \
    --plot

import tempfile, os as _os
from IPython.display import Image
Image(_os.path.join(tempfile.gettempdir(), 'fom_solution.png'))

### 2.2 Generazione snapshot (training + test)

Skip se i file esistono gia'. Seed diverso per il test set.

In [ ]:
N_SAMPLES = 300
N_TEST = 150
TEST_SEED = 123  # diverso dal seed di default usato per il training (42)

SNAPSHOTS_PATH = os.path.join(SNAPSHOTS_DIR, f'{TEST}_{N_SAMPLES}.npz')
TEST_SNAPSHOTS_PATH = os.path.join(SNAPSHOTS_DIR, f'{TEST}_test{N_TEST}.npz')

if os.path.exists(SNAPSHOTS_PATH):
    print(f"Training set gia' presente: {SNAPSHOTS_PATH} (skip)")
else:
    !python -m src.full_order.generate_snapshots \
        --config {CONFIG_PATH} \
        --n-samples {N_SAMPLES} \
        --output {SNAPSHOTS_PATH}

if os.path.exists(TEST_SNAPSHOTS_PATH):
    print(f"Test set gia' presente: {TEST_SNAPSHOTS_PATH} (skip)")
else:
    !python -m src.full_order.generate_snapshots \
        --config {CONFIG_PATH} \
        --n-samples {N_TEST} \
        --seed {TEST_SEED} \
        --output {TEST_SNAPSHOTS_PATH}

In [ ]:
import numpy as np

data = np.load(SNAPSHOTS_PATH)
print('Training - chiavi:', list(data.keys()), ' Y shape:', data['Y'].shape)

test_data = np.load(TEST_SNAPSHOTS_PATH)
print('Test     - chiavi:', list(test_data.keys()), ' Y shape:', test_data['Y'].shape)

## 3. ROM

### 3.1 POD

In [ ]:
INNER_PRODUCT = 'seminorm'
ENERGY_THRESHOLD = 1-1e-6   # N scelto automaticamente in base a questa soglia
NORMALIZE_CORRELATION = True
POD_PATH = os.path.join(SNAPSHOTS_DIR, f'{TEST}_pod.npz')

normalize_flag = '' if NORMALIZE_CORRELATION else '--no-normalize-correlation'

!python -m src.rom.train_pod \
    --config {CONFIG_PATH} \
    --snapshots {SNAPSHOTS_PATH} \
    --test-snapshots {TEST_SNAPSHOTS_PATH} \
    --inner-product {INNER_PRODUCT} \
    --energy-threshold {ENERGY_THRESHOLD} \
    {normalize_flag} \
    --output {POD_PATH} \
    --plot

import tempfile, os as _os
from IPython.display import Image
Image(_os.path.join(tempfile.gettempdir(), 'pod_eigenvalue_decay.png'))

### 3.2 Sweep PODNN

Nel paper (Test 2) servono molti piu' modi che nel Test 1 (stato: errore ~1e-2 con N=80), quindi il range e' esteso. Skip se i csv esistono.

In [ ]:
N_MODES_VALUES = '10,20,30,40,60,80'
HIDDEN_DIM = 32
SWEEP_EPOCHS = 150000
SWEEP_N_MODES_OUTPUT = os.path.join(SNAPSHOTS_DIR, 'sweep_n_modes.csv')
SWEEP_POD_ERROR_OUTPUT = os.path.join(SNAPSHOTS_DIR, 'sweep_pod_error.csv')

if os.path.exists(SWEEP_N_MODES_OUTPUT):
    print(f"Sweep PODNN gia' presente: {SWEEP_N_MODES_OUTPUT} (skip - cancella il file per rilanciarlo)")
else:
    !python -m src.rom.sweep_pod_nn \
        --config {CONFIG_PATH} \
        --snapshots {SNAPSHOTS_PATH} \
        --test-snapshots {TEST_SNAPSHOTS_PATH} \
        --sweep-param n_modes \
        --values {N_MODES_VALUES} \
        --hidden-dim {HIDDEN_DIM} \
        --epochs {SWEEP_EPOCHS} \
        --output {SWEEP_N_MODES_OUTPUT}

if os.path.exists(SWEEP_POD_ERROR_OUTPUT):
    print(f"Sweep solo-POD gia' presente: {SWEEP_POD_ERROR_OUTPUT} (skip - cancella il file per rilanciarlo)")
else:
    !python -m src.rom.sweep_pod_error \
        --config {CONFIG_PATH} \
        --snapshots {SNAPSHOTS_PATH} \
        --test-snapshots {TEST_SNAPSHOTS_PATH} \
        --values {N_MODES_VALUES} \
        --podnn-csv {SWEEP_N_MODES_OUTPUT} \
        --output {SWEEP_POD_ERROR_OUTPUT}
from pathlib import Path
from IPython.display import Image
Image(str(Path(SWEEP_POD_ERROR_OUTPUT).with_suffix('.png')))

### 3.3 Training finale PODNN (stato + aggiunto)

In [ ]:
PODNN_EPOCHS = 150000
PODNN_HIDDEN_DIM = 32
N_MODES_Y = None  # None = scelti dalla POD (soglia energia); poi aggiorna guardando lo sweep sopra
N_MODES_P = None

Y_NN_PATH = os.path.join(SNAPSHOTS_DIR, f'{TEST}_y_nn.pt')
P_NN_PATH = os.path.join(SNAPSHOTS_DIR, f'{TEST}_p_nn.pt')

def train_podnn_if_missing(field, output_path, n_modes):
    if os.path.exists(output_path):
        print(f"{output_path} gia' presente (skip - cancella {output_path} e il suo .norm.npz per rilanciare)")
        return
    n_modes_flag = f'--n-modes {n_modes}' if n_modes is not None else ''
    !python -m src.rom.train_reduced_nn \
        --pod-model {POD_PATH} \
        --field {field} \
        --epochs {PODNN_EPOCHS} \
        --hidden-dim {PODNN_HIDDEN_DIM} \
        {n_modes_flag} \
        --output {output_path}

train_podnn_if_missing('y', Y_NN_PATH, N_MODES_Y)
train_podnn_if_missing('p', P_NN_PATH, N_MODES_P)

### 3.4 Valutazione PODNN (errore + speedup)

In [ ]:
!python -m src.rom.evaluate_pod_nn \
    --config {CONFIG_PATH} \
    --pod-model {POD_PATH} \
    --weights-y {Y_NN_PATH} \
    --weights-p {P_NN_PATH} \
    --test-snapshots {TEST_SNAPSHOTS_PATH}

## 4. GNN

Basata su [gca-rom](https://github.com/fpichi/gca-rom) (non modificata): solo `y`, solo `p`, combinato.

### 4.0 Setup

In [ ]:
GCA_ROM_PATH = 'gca-rom'  # <- path alla repo gca-rom clonata (aggiorna se diverso)
GNN_DIR = os.path.join(DATA_DIR, 'gnn')
os.makedirs(GNN_DIR, exist_ok=True)

import importlib.util
if importlib.util.find_spec('torch_geometric') is None:
    !pip install -q torch_geometric

if not os.path.isdir(GCA_ROM_PATH):
    !git clone https://github.com/fpichi/gca-rom.git {GCA_ROM_PATH}
else:
    print(f"{GCA_ROM_PATH} gia' presente (skip clone)")

### 4.1 Conversione snapshot -> formato gca-rom

In [ ]:
def convert_if_missing(snapshots_path, output_path, n_comp, field=None):
    if os.path.exists(output_path):
        print(f"{output_path} gia' presente (skip)")
        return
    field_flag = f'--field {field}' if field is not None else ''
    !python -m src.gnn.convert_to_gca_rom \
        --config {CONFIG_PATH} \
        --snapshots {snapshots_path} \
        --n-comp {n_comp} \
        {field_flag} \
        --output {output_path}

TRAIN_Y_MAT = os.path.join(GNN_DIR, 'train_y.mat')
TEST_Y_MAT = os.path.join(GNN_DIR, 'test_y.mat')
TRAIN_P_MAT = os.path.join(GNN_DIR, 'train_p.mat')
TEST_P_MAT = os.path.join(GNN_DIR, 'test_p.mat')
TRAIN_YP_MAT = os.path.join(GNN_DIR, 'train_yp.mat')
TEST_YP_MAT = os.path.join(GNN_DIR, 'test_yp.mat')

convert_if_missing(SNAPSHOTS_PATH, TRAIN_Y_MAT, n_comp=1, field='y')
convert_if_missing(TEST_SNAPSHOTS_PATH, TEST_Y_MAT, n_comp=1, field='y')
convert_if_missing(SNAPSHOTS_PATH, TRAIN_P_MAT, n_comp=1, field='p')
convert_if_missing(TEST_SNAPSHOTS_PATH, TEST_P_MAT, n_comp=1, field='p')
convert_if_missing(SNAPSHOTS_PATH, TRAIN_YP_MAT, n_comp=2)
convert_if_missing(TEST_SNAPSHOTS_PATH, TEST_YP_MAT, n_comp=2)

### 4.2 Training: solo y, solo p, combinato

Skip se `train_meta.json` esiste con gli stessi parametri (cancella la cartella della rete per rilanciare).

In [ ]:
import json
import shutil

GNN_EPOCHS = 5000
GNN_BOTTLENECK = 15
GNN_BATCH_SIZE = None  # None = full-batch; es. 32 per mini-batch
GNN_MINIBATCH = False  # richiede GNN_BATCH_SIZE impostato
GNN_MODELS_DIR = os.path.join(GNN_DIR, 'models')

def train_if_missing(train_mat, test_mat, comp, net_name, field=None):
    net_dir = os.path.join(GNN_MODELS_DIR, net_name)
    meta_path = os.path.join(net_dir, 'train_meta.json')
    target = {'epochs': GNN_EPOCHS, 'batch_size': GNN_BATCH_SIZE, 'minibatch': GNN_MINIBATCH}

    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        if all(meta.get(k) == v for k, v in target.items()):
            print(f"{meta_path} gia' presente con parametri identici (skip)")
            return
        print(f"{meta_path} presente ma con parametri diversi - ri-alleno")
    if os.path.exists(net_dir):
        shutil.rmtree(net_dir)

    field_flag = f'--field {field}' if field is not None else ''
    batch_flag = f'--batch-size {GNN_BATCH_SIZE}' if GNN_BATCH_SIZE is not None else ''
    minibatch_flag = '--minibatch' if (GNN_BATCH_SIZE is not None and GNN_MINIBATCH) else ''
    !python -m src.gnn.train_gnn \
        --config {CONFIG_PATH} \
        --gca-rom-path {GCA_ROM_PATH} \
        --train-mat {train_mat} \
        --test-mat {test_mat} \
        --comp {comp} \
        {field_flag} \
        --net-name {net_name} \
        --net-dir {net_dir} \
        --bottleneck-dim {GNN_BOTTLENECK} \
        --epochs {GNN_EPOCHS} \
        {batch_flag} \
        {minibatch_flag}

train_if_missing(TRAIN_Y_MAT, TEST_Y_MAT, comp=1, net_name=f'{TEST}_gnn_y', field='y')
train_if_missing(TRAIN_P_MAT, TEST_P_MAT, comp=1, net_name=f'{TEST}_gnn_p', field='p')
train_if_missing(TRAIN_YP_MAT, TEST_YP_MAT, comp=2, net_name=f'{TEST}_gnn_yp')

### 4.3 Valutazione (errore L2/H1 + speedup)

In [ ]:
def evaluate(train_mat, test_mat, net_name):
    net_dir = os.path.join(GNN_MODELS_DIR, net_name)
    !python -m src.gnn.evaluate_gnn \
        --config {CONFIG_PATH} \
        --gca-rom-path {GCA_ROM_PATH} \
        --train-mat {train_mat} \
        --test-mat {test_mat} \
        --net-dir {net_dir}

evaluate(TRAIN_Y_MAT, TEST_Y_MAT, f'{TEST}_gnn_y')
evaluate(TRAIN_P_MAT, TEST_P_MAT, f'{TEST}_gnn_p')
evaluate(TRAIN_YP_MAT, TEST_YP_MAT, f'{TEST}_gnn_yp')

### 4.4 Valutazione su un singolo punto (mu1, mu2, mu_u)

In [ ]:
NET_NAME = f'{TEST}_gnn_yp'
TRAIN_MAT = TRAIN_YP_MAT
TEST_MAT = TEST_YP_MAT
MU1 = 12
MU2 = 2.5
MU_U = 0.5

NET_DIR = os.path.join(GNN_MODELS_DIR, NET_NAME)

!python -m src.gnn.evaluate_gnn_single \
    --config {CONFIG_PATH} \
    --gca-rom-path {GCA_ROM_PATH} \
    --train-mat {TRAIN_MAT} \
    --test-mat {TEST_MAT} \
    --net-dir {NET_DIR} \
    --mu1 {MU1} \
    --mu2 {MU2} \
    --mu_u {MU_U} \
    --plot

import tempfile, os as _os
from IPython.display import Image
Image(_os.path.join(tempfile.gettempdir(), 'gnn_single_solution.png'))

## 5. Controllo (POD)

`u` e' una traccia 1D sui nodi del bordo del delfino (marker 4), mascherata a zero dove `x < x_ctrl = mu_u` (fuori da Gamma_C). POD con prodotto scalare euclideo: l'ordine dei nodi di bordo non influisce sulla base.

In [ ]:
CONTROL_POD_PATH = os.path.join(SNAPSHOTS_DIR, f'{TEST}_control_pod.npz')
CONTROL_ENERGY_THRESHOLD = 1 - 1e-6
CONTROL_MAX_MODES = 15
CONTROL_NORMALIZE_CORRELATION = True

control_normalize_flag = '' if CONTROL_NORMALIZE_CORRELATION else '--no-normalize-correlation'

!python -m src.rom.build_control_pod \
    --config {CONFIG_PATH} \
    --snapshots {SNAPSHOTS_PATH} \
    --test-snapshots {TEST_SNAPSHOTS_PATH} \
    --energy-threshold {CONTROL_ENERGY_THRESHOLD} \
    --max-modes {CONTROL_MAX_MODES} \
    {control_normalize_flag} \
    --output {CONTROL_POD_PATH} \
    --plot

import tempfile, os as _os
from IPython.display import Image
Image(_os.path.join(tempfile.gettempdir(), 'control_pod_eigenvalue_decay.png'))